# Pasarela Colab para motor OCR existente\nNo instala el motor original. Requiere un backend OCR activo y accesible desde Colab.\nEjecuta las celdas en orden y después configura el cliente Windows.

In [ ]:
!pip -q install fastapi uvicorn httpx python-multipart
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null


In [ ]:
import requests

url = (
    "https://hub.docker.com/v2/repositories/"
    "reijohan/unlimited-ocr2-vast/tags/runtime/"
)

r = requests.get(url, timeout=20)

print("HTTP:", r.status_code)

if r.ok:
    data = r.json()
    print("Etiqueta:", data.get("name"))
    print("Tamaño:", round(data.get("full_size", 0) / 1024**3, 2), "GB")
else:
    print("La imagen no está disponible públicamente en Docker Hub.")

HTTP: 200
Etiqueta: runtime
Tamaño: 8.82 GB


In [ ]:
import subprocess
import shutil

print("=== GPU ===")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])
else:
    print("No hay GPU NVIDIA disponible")

print("\n=== ESPACIO EN DISCO ===")
total, used, free = shutil.disk_usage("/")
print(f"Libre: {free / 1024**3:.2f} GB")

print("\n=== MEMORIA RAM ===")
with open("/proc/meminfo") as f:
    for line in f:
        if line.startswith(("MemTotal:", "MemAvailable:")):
            print(line.strip())

=== GPU ===

=== ESPACIO EN DISCO ===
Libre: 112.54 GB

=== MEMORIA RAM ===
MemTotal:       13286932 kB
MemAvailable:   11968980 kB


In [ ]:
import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("Colab sigue ejecutándose sin GPU CUDA")

CUDA disponible: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
import requests

repo = "reijohan/unlimited-ocr2-vast"
tag = "runtime"

token_response = requests.get(
    "https://auth.docker.io/token",
    params={
        "service": "registry.docker.io",
        "scope": f"repository:{repo}:pull"
    },
    timeout=30
)
token_response.raise_for_status()

token = token_response.json()["token"]

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": (
        "application/vnd.oci.image.index.v1+json,"
        "application/vnd.docker.distribution.manifest.list.v2+json,"
        "application/vnd.oci.image.manifest.v1+json,"
        "application/vnd.docker.distribution.manifest.v2+json"
    )
}

url = f"https://registry-1.docker.io/v2/{repo}/manifests/{tag}"

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

manifest = response.json()

print("TIPO:", manifest.get("mediaType"))

if "manifests" in manifest:
    for item in manifest["manifests"]:
        print("PLATAFORMA:", item.get("platform"))
        print("DIGEST:", item.get("digest"))
else:
    print("CONFIG:", manifest.get("config"))
    print("CAPAS:", len(manifest.get("layers", [])))

TIPO: application/vnd.oci.image.index.v1+json
PLATAFORMA: {'architecture': 'amd64', 'os': 'linux'}
DIGEST: sha256:9b65b3c50be1e4012bad709f38f9ed17359b5ec615c5d2fc606588512254a59a
PLATAFORMA: {'architecture': 'unknown', 'os': 'unknown'}
DIGEST: sha256:ca86544a0cf00de4346638818366248338e053b2b7f565649ce900fdf14c0fd2


In [ ]:
import requests
import json

repo = "reijohan/unlimited-ocr2-vast"
digest = "sha256:9b65b3c50be1e4012bad709f38f9ed17359b5ec615c5d2fc606588512254a59a"

token = requests.get(
    "https://auth.docker.io/token",
    params={
        "service": "registry.docker.io",
        "scope": f"repository:{repo}:pull"
    },
    timeout=30
).json()["token"]

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": (
        "application/vnd.oci.image.manifest.v1+json,"
        "application/vnd.docker.distribution.manifest.v2+json"
    )
}

base = f"https://registry-1.docker.io/v2/{repo}"

r = requests.get(
    f"{base}/manifests/{digest}",
    headers=headers,
    timeout=30
)
r.raise_for_status()

manifest = r.json()

print("CAPAS:", len(manifest.get("layers", [])))

config_digest = manifest["config"]["digest"]

config_response = requests.get(
    f"{base}/blobs/{config_digest}",
    headers={"Authorization": f"Bearer {token}"},
    timeout=60
)
config_response.raise_for_status()

config = config_response.json().get("config", {})

print("\n=== CONFIGURACIÓN DEL MOTOR ===")

for campo in ["Entrypoint", "Cmd", "WorkingDir", "ExposedPorts"]:
    print(f"{campo}:", config.get(campo))

print("\n=== VARIABLES DE ENTORNO ===")

for variable in config.get("Env", []):
    nombre = variable.split("=", 1)[0]
    print(nombre)

CAPAS: 20

=== CONFIGURACIÓN DEL MOTOR ===
Entrypoint: ['/usr/bin/tini', '--', '/app/start.sh']
Cmd: None
WorkingDir: /app
ExposedPorts: {'10001/tcp': {}}

=== VARIABLES DE ENTORNO ===
PATH
NVARCH
NVIDIA_REQUIRE_CUDA
NV_CUDA_CUDART_VERSION
CUDA_VERSION
LD_LIBRARY_PATH
NVIDIA_VISIBLE_DEVICES
NVIDIA_DRIVER_CAPABILITIES
NV_CUDA_LIB_VERSION
NV_NVTX_VERSION
NV_LIBNPP_VERSION
NV_LIBNPP_PACKAGE
NV_LIBCUSPARSE_VERSION
NV_LIBCUBLAS_PACKAGE_NAME
NV_LIBCUBLAS_VERSION
NV_LIBCUBLAS_PACKAGE
NV_LIBNCCL_PACKAGE_NAME
NV_LIBNCCL_PACKAGE_VERSION
NCCL_VERSION
NV_LIBNCCL_PACKAGE
NVIDIA_PRODUCT_NAME
NV_CUDA_CUDART_DEV_VERSION
NV_NVML_DEV_VERSION
NV_LIBCUSPARSE_DEV_VERSION
NV_LIBNPP_DEV_VERSION
NV_LIBNPP_DEV_PACKAGE
NV_LIBCUBLAS_DEV_VERSION
NV_LIBCUBLAS_DEV_PACKAGE_NAME
NV_LIBCUBLAS_DEV_PACKAGE
NV_CUDA_NSIGHT_COMPUTE_VERSION
NV_CUDA_NSIGHT_COMPUTE_DEV_PACKAGE
NV_NVPROF_VERSION
NV_NVPROF_DEV_PACKAGE
NV_LIBNCCL_DEV_PACKAGE_NAME
NV_LIBNCCL_DEV_PACKAGE_VERSION
NV_LIBNCCL_DEV_PACKAGE
LIBRARY_PATH
PYTHONUNBUFFERED

In [ ]:
import requests

repo = "reijohan/unlimited-ocr2-vast"
digest = "sha256:9b65b3c50be1e4012bad709f38f9ed17359b5ec615c5d2fc606588512254a59a"

token = requests.get(
    "https://auth.docker.io/token",
    params={
        "service": "registry.docker.io",
        "scope": f"repository:{repo}:pull"
    },
    timeout=30
).json()["token"]

base = f"https://registry-1.docker.io/v2/{repo}"

manifest_response = requests.get(
    f"{base}/manifests/{digest}",
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.oci.image.manifest.v1+json"
    },
    timeout=30
)
manifest_response.raise_for_status()

config_digest = manifest_response.json()["config"]["digest"]

config_response = requests.get(
    f"{base}/blobs/{config_digest}",
    headers={"Authorization": f"Bearer {token}"},
    timeout=60
)
config_response.raise_for_status()

config = config_response.json()["config"]

variables = dict(
    item.split("=", 1)
    for item in config.get("Env", [])
    if "=" in item
)

campos = [
    "OCR_MODEL_ID",
    "CUDA_VERSION",
    "OCR_PORT",
    "SGLANG_MEM_FRACTION_STATIC",
    "SGLANG_MAX_RUNNING_REQUESTS",
    "SGLANG_UOCR_MAX_PAGE_TOKENS",
    "MAX_PAGES",
    "DEFAULT_DPI"
]

print("=== CONFIGURACIÓN DEL MOTOR OCR ===")

for campo in campos:
    print(f"{campo}: {variables.get(campo, 'No definido')}")

=== CONFIGURACIÓN DEL MOTOR OCR ===
OCR_MODEL_ID: baidu/Unlimited-OCR-2
CUDA_VERSION: 12.8.1
OCR_PORT: 10001
SGLANG_MEM_FRACTION_STATIC: 0.70
SGLANG_MAX_RUNNING_REQUESTS: 4
SGLANG_UOCR_MAX_PAGE_TOKENS: 20000
MAX_PAGES: 12
DEFAULT_DPI: 200


In [ ]:
import torch
import subprocess

print("=== GPU ===")
print("Modelo:", torch.cuda.get_device_name(0))
print("Arquitectura CUDA:", torch.cuda.get_device_capability(0))
print("BF16 soportado:", torch.cuda.is_bf16_supported())
print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)

print("\n=== CONTROLADOR NVIDIA ===")
subprocess.run(["nvidia-smi"], check=False)

=== GPU ===
Modelo: Tesla T4
Arquitectura CUDA: (7, 5)
BF16 soportado: True
PyTorch: 2.11.0+cu130
CUDA de PyTorch: 13.0

=== CONTROLADOR NVIDIA ===


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
import requests

modelo = "baidu/Unlimited-OCR-2"
url = f"https://huggingface.co/api/models/{modelo}"

r = requests.get(url, timeout=30)
r.raise_for_status()
data = r.json()

print("=== MODELO ===")
print("ID:", data["id"])
print("Arquitectura:", data.get("config", {}).get("architectures"))
print("Dtype:", data.get("config", {}).get("torch_dtype"))

print("\n=== PESOS ===")
total = 0
for archivo in data.get("siblings", []):
    nombre = archivo["rfilename"]
    if nombre.endswith((".safetensors", ".bin")):
        print(nombre)
        total += archivo.get("size", 0) or 0

if total:
    print("Tamaño:", round(total / 1024**3, 2), "GiB")
else:
    print("La API no proporciona tamaños completos.")

print("\n=== GPU ===")
import torch
print(torch.cuda.get_device_name(0))
print("VRAM:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GiB")

=== MODELO ===
ID: baidu/Unlimited-OCR-2
Arquitectura: ['UnlimitedOCRForCausalLM']
Dtype: None

=== PESOS ===
model-00001-of-00002.safetensors
model-00002-of-00002.safetensors
La API no proporciona tamaños completos.

=== GPU ===
Tesla T4
VRAM: 14.56 GiB


In [ ]:
import requests

modelo = "baidu/Unlimited-OCR-2"
url = f"https://huggingface.co/api/models/{modelo}/tree/main"

r = requests.get(
    url,
    params={"recursive": "true", "expand": "true"},
    timeout=60
)
r.raise_for_status()

total = 0

for archivo in r.json():
    nombre = archivo.get("path", "")

    if nombre.endswith(".safetensors"):
        size = archivo.get("size", 0)
        total += size

        print(
            nombre,
            f"{size / 1024**3:.2f} GiB"
        )

print("\nTOTAL DE PESOS:")
print(f"{total / 1024**3:.2f} GiB")

print("\nVRAM DISPONIBLE: 14.56 GiB")

model-00001-of-00002.safetensors 4.66 GiB
model-00002-of-00002.safetensors 1.57 GiB

TOTAL DE PESOS:
6.23 GiB

VRAM DISPONIBLE: 14.56 GiB


In [ ]:
%pip -q install -U huggingface_hub

from huggingface_hub import snapshot_download

ruta = snapshot_download(
    repo_id="baidu/Unlimited-OCR-2",
    local_dir="/content/Unlimited-OCR-2"
)

print("Modelo descargado en:", ruta)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Modelo descargado en: /content/Unlimited-OCR-2


In [ ]:
%pip install -q --upgrade --upgrade-strategy eager \
    "huggingface-hub>=1.32,<3" \
    "datasets>=4.8.5" \
    "transformers==5.22.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.7.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.107 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requir

In [ ]:
import os, secrets
from getpass import getpass
# Usa una clave nueva para el cliente -> Colab; no la publiques.
os.environ["OCR_API_KEY"] = getpass("Clave nueva para el cliente OCR: ").strip()
# URL del MOTOR EXISTENTE, por ejemplo http://servidor:10001
os.environ["OCR_BACKEND_URL"] = input("URL del motor OCR activo: ").strip().rstrip("/")
os.environ["OCR_BACKEND_API_KEY"] = getpass("Clave del motor OCR existente: ").strip()
assert os.environ["OCR_API_KEY"] and os.environ["OCR_BACKEND_URL"]


In [ ]:
from google.colab import files
print("Sube el archivo colab_proxy.py incluido en el ZIP")
files.upload()


In [ ]:
import subprocess,time,re,os
api=subprocess.Popen(["uvicorn","colab_proxy:app","--host","127.0.0.1","--port","8000"],stdout=open('/tmp/ocr_api.log','w'),stderr=subprocess.STDOUT)
time.sleep(3)
tunnel_log=open('/tmp/ocr_tunnel.log','w')
tunnel=subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
url=None
for _ in range(30):
    time.sleep(1)
    txt=open('/tmp/ocr_tunnel.log').read()
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',txt)
    if m: url=m.group(0);break
if not url: print('No se pudo obtener URL. Revisa /tmp/ocr_tunnel.log')
else: print('COLAB_OCR_URL='+url+'\nCopia esta URL en tu .env local. Mantén la sesión de Colab abierta.')
